# المرحلة الأولى: استيراد المكتبات وتحميل البيانات
هنا نقوم باستدعاء المكتبات الأساسية:
* **Pandas**: للتعامل مع جدول البيانات.
* **NumPy**: للعمليات الحسابية على المصفوفات.
* **Matplotlib**: للرسم البياني (إذا لزم الأمر).

سنقوم بقراءة الملف `Churn_Modelling_2.csv` وعرض أول صفوف منه.

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# تحميل الملف (تأكد من رفعه في Files على اليسار بنفس الاسم تماماً)
dataset = pd.read_csv('Churn_Modelling.csv')

# عرض عينة من البيانات
dataset.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


# المرحلة الثانية: اختيار الميزات (Features) والهدف (Target)
في هذه الخطوة نقسم البيانات إلى:
1. **X (الميزات)**: المعلومات التي سيتعلم منها الموديل.
   - قمنا بحذف أول 3 أعمدة (رقم الصف، معرف العميل، اللقب) لأنها بيانات "إدارية" لا تؤثر على قرار العميل بالبقاء أو الرحيل.
2. **y (الهدف)**: وهو العمود الأخير الذي يخبرنا هل رحل العميل (1) أم بقي (0).

In [21]:
# الميزات من العمود الرابع (الترتيب 3) حتى قبل الأخير
X = dataset.iloc[:, 3:-1].values

# الهدف هو العمود الأخير
y = dataset.iloc[:, -1].values

print("أول صف من الميزات (بدون البيانات الإدارية):\n", X[0])

أول صف من الميزات (بدون البيانات الإدارية):
 [619 'France' 'Female' 42 2 0.0 1 1 1 101348.88]


# المرحلة الثالثة: تحويل النصوص إلى أرقام (Encoding)
بما أن الشبكة العصبية لا تفهم إلا الأرقام، سنقوم بـ:
1. تحويل عمود **النوع (Gender)** إلى (0 و 1) باستخدام `LabelEncoder`.
2. تحويل عمود **الدولة (Geography)** إلى أعمدة منفصلة باستخدام `OneHotEncoder` لتجنب إعطاء ترتيب رقمي خاطئ للدول.

In [22]:
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# تحويل النوع (العمود رقم 2 في مصفوفة X)
le = LabelEncoder()
X[:, 2] = le.fit_transform(X[:, 2])

# تحويل الدولة (العمود رقم 1 في مصفوفة X)
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [1])], remainder='passthrough')
X = np.array(ct.fit_transform(X))

print("أول صف بعد تحويل النصوص لأرقام:\n", X[0])

أول صف بعد تحويل النصوص لأرقام:
 [1.0 0.0 0.0 619 0 42 2 0.0 1 1 1 101348.88]


# المرحلة الرابعة: تقسيم البيانات وموازنتها (Scaling)
1. **التقسيم**: نقسم البيانات لـ 80% تدريب (يتعلم منها الموديل) و 20% اختبار (نمتحنه بها).
2. **الموازنة (Feature Scaling)**: خطوة "حيوية" للـ ANN. الأرقام في الجدول متفاوتة (الراتب بالآلاف، السن بالعشرات).
   - نستخدم `StandardScaler` لجعل كل الأرقام في نطاق متقارب (حول الصفر) حتى لا تسيطر الأرقام الكبيرة على الحسابات.

In [23]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# التقسيم
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# الموازنة (Scaling)
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

print("بيانات التدريب بعد الموازنة:\n", X_train[0])

بيانات التدريب بعد الموازنة:
 [-1.01460667 -0.5698444   1.74309049  0.16958176 -1.09168714 -0.46460796
  0.00666099 -1.21571749  0.8095029   0.64259497 -1.03227043  1.10643166]


# المرحلة الخامسة: بناء هيكل الشبكة العصبية (Building the ANN)
في هذه المرحلة، نستخدم مكتبة **Keras** (التابعة لـ TensorFlow) لبناء الموديل كطبقات متتالية:

1.  **Initializing**: بنعرف الموديل إنه `Sequential` (يعني طبقات ورا بعض).
2.  **Input & Hidden Layer**: بنضيف أول طبقة "مخفية"، وبنحدد لها عدد الخلايا العصبية (مثلاً 6). دالة التنشيط هنا هي **ReLU**، وهي المسؤولة عن تمرير الإشارات القوية فقط.
3.  **Second Hidden Layer**: بنضيف طبقة تانية لزيادة قدرة الموديل على التعلم.
4.  **Output Layer**: الطبقة الأخيرة، ولازم يكون فيها خلية واحدة بس (لأننا بنوقع نتيحة واحدة: يمشي أو يقعد). بنستخدم هنا دالة **Sigmoid** لأنها بتطلع النتيجة كنسبة مئوية (احتمالية).

In [24]:
import tensorflow as tf

# 1. تهيئة الشبكة العصبية كـ Sequential
ann = tf.keras.models.Sequential()

# 2. إضافة طبقة المدخلات والطبقة المخفية الأولى
# units=6 (عدد العصبونات)، activation='relu' (دالة التنشيط)
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))

# 3. إضافة الطبقة المخفية الثانية
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))

# 4. إضافة طبقة المخرجات
# استخدمنا sigmoid عشان تطلع احتمال بين 0 و 1
ann.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

# المرحلة السادسة: تجميع وتدريب الموديل (Compiling & Training)
بعد ما بنينا الهيكل، لازم نحدد "طريقة المذاكرة":

1.  **Optimizer**: بنختار `adam` لأنه أذكى طريقة لتحديث الأوزان (Weights) عشان الموديل يقلل غلطه بسرعة.
2.  **Loss Function**: بنختار `binary_crossentropy` لأننا بنحل مشكلة تصنيف ثنائي (اه أو لأ).
3.  **Training**: بنبدأ التدريب الفعلي باستخدام `fit`.
    - **Epochs**: عدد المرات اللي الموديل هيقرأ فيها الداتا كلها (مثلاً 100 مرة).
    - **Batch Size**: عدد العينات اللي بيشوفها الموديل قبل ما يحدّث أوزانه.

In [25]:
# تجميع الموديل
ann.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

# تدريب الموديل على بيانات التدريب
# هتشوف الموديل وهو بيتعلم قدامك والدقة (Accuracy) بتزيد
ann.fit(X_train, y_train, batch_size = 32, epochs = 100)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7788 - loss: 0.5063
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7980 - loss: 0.4447
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8096 - loss: 0.4320
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8155 - loss: 0.4255
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8173 - loss: 0.4216
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8196 - loss: 0.4186
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8249 - loss: 0.4161
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8242 - loss: 0.4147
Epoch 9/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8257 - loss: 0.4129
Epoch 10/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8261 - loss: 0.4114
Epoch 11/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8289 - loss: 0.4099
Epoch 12/100
250/250 ━━━━━━━━━━━━━━━━━━━━

In [ ]:
# --- خلية التنبؤ التفاعلي ---

import numpy as np

print("================================================")
print("   نظام التنبؤ برحيل العملاء (ANN Churn Predictor)   ")
print("================================================\n")

try:
    # 1. استقبال البيانات من المستخدم
    print("الرجاء إدخال بيانات العميل الجديد:")
    credit_score = float(input("- رصيد النقاط الائتمانية (مثلاً 600): "))
    geography = input("- الدولة (France / Germany / Spain): ").strip().capitalize()
    gender = input("- النوع (Male / Female): ").strip().capitalize()
    age = int(input("- السن: "))
    tenure = int(input("- عدد سنوات الولاء للبنك: "))
    balance = float(input("- رصيد الحساب الحالي: "))
    num_of_products = int(input("- عدد المنتجات المشترك بها (1-4): "))
    has_cr_card = int(input("- هل لديه بطاقة ائتمان؟ (نعم=1 / لا=0): "))
    is_active_member = int(input("- هل هو عضو نشط؟ (نعم=1 / لا=0): "))
    estimated_salary = float(input("- الراتب السنوي المتوقع: "))

    # 2. التشفير اليدوي (Manual Encoding) ليتناسب مع هيكل X_train
    # ترتيب أعمدة الدولة بعد الـ OneHotEncoder: [France, Germany, Spain]
    if geography == 'France':
        geo_encoded = [1, 0, 0]
    elif geography == 'Germany':
        geo_encoded = [0, 1, 0]
    else: # Spain
        geo_encoded = [0, 0, 1]

    # تشفير النوع (LabelEncoder كان قد حول Female=0, Male=1 غالباً)
    gender_encoded = 1 if gender == 'Male' else 0

    # 3. تجميع كافة الميزات في مصفوفة واحدة بنفس الترتيب الصحيح
    # الترتيب: Geography(3 cols), CreditScore, Gender, Age, Tenure, Balance, NumOfProducts, HasCrCard, IsActiveMember, EstimatedSalary
    new_customer = geo_encoded + [credit_score, gender_encoded, age, tenure, balance, num_of_products, has_cr_card, is_active_member, estimated_salary]

    # 4. تحويل البيانات إلى مصفوفة NumPy وعمل Scaling
    new_customer_scaled = sc.transform([new_customer])

    # 5. إجراء التنبؤ
    prediction_prob = ann.predict(new_customer_scaled)[0][0]
    will_leave = prediction_prob > 0.5

    # 6. عرض النتائج بشكل احترافي
    print("\n" + "="*30)
    print(f"تحليل البيانات...")
    print(f"نسبة احتمالية رحيل العميل: {prediction_prob * 100:.2f}%")

    if will_leave:
        print("القرار النهائي: 🚩 العميل سيترك البنك غالباً (EXIT)")
    else:
        print("القرار النهائي: ✅ العميل سيبقى في البنك (STAY)")
    print("="*30)

except ValueError:
    print("\n❌ خطأ: يرجى إدخال أرقام صحيحة في الحقول المطلوبة.")
except Exception as e:
    print(f"\n❌ حدث خطأ غير متوقع: {e}")

   نظام التنبؤ برحيل العملاء (ANN Churn Predictor)   

الرجاء إدخال بيانات العميل الجديد:
